# Microsoft Agent Çerçevesi — Azure OpenAI (Responses API)

Bu kod örneğinde, **Responses API** kullanarak **Azure OpenAI** tarafından desteklenen basit bir ajan oluşturmak için **Microsoft Agent Çerçevesi (MAF)** kullanacaksınız.

> **Geçiş notu:** Bu örnek daha önce GitHub Modelleri ile Semantic Kernel kullanıyordu. Microsoft Agent Çerçevesine taşındı ve GitHub Modelleri (kullanımdan kaldırılıyor, Temmuz 2026'da emekliye ayrılacak) Azure OpenAI ile değiştirildi; Azure OpenAI Responses API'yi destekliyor. MAF'deki `OpenAIChatClient`, Azure OpenAI’nin kararlı `/openai/v1/` uç noktasını hedefler ve varsayılan olarak Responses API'yi kullanır.

Bu örneğin amacı, çeşitli ajan modelleri uygulanırken sonraki kod örneklerinde uygulanacak adımları göstermektir.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Gerekli Python Paketlerini İçe Aktar


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Bir Araç Tanımlamak

Microsoft Agent Framework'te, **araç** ajan tarafından çağrılabilen `@tool` ile dekorasyon yapılmış basit bir Python fonksiyonudur. Aşağıda rastgele bir tatil destinasyonu döndüren ve önceki destinasyonu tekrarlamaktan kaçınan bir araç tanımlıyoruz.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Creating the Agent

Here, we create the Agent named `TravelAgent`.

In this example, we use very basic instructions. Feel free to modify these instructions to observe how the agent's behavior changes.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Ajanı Çalıştırma

Artık ajanı çalıştırabiliriz. Ajanın dönüşler arasında sohbeti hatırlaması için bir `AgentSession` oluşturuyoruz, sonra iki `user_inputs` gönderiyoruz. İlkinde bir gezi isteniyor; ikincisinde kullanıcı öneriyi beğenmediğini söylüyor ve başka bir tane istiyor — ajan, cevap verirken oturum geçmişini ve `get_random_destination` aracını kullanıyor.

Bu mesajları, ajanın farklı tepki vermesini gözlemlemek için değiştirebilirsiniz. Yanıtlar **token token** akış halinde gönderilir.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
